## Collaborative Filtering Cost Function

In [1]:
import numpy as np

def cofi_cost_func(X, W, b, Y, R, lambda_):
    """
    Collaborative filtering cost function
    """
    nm, nu = Y.shape
    J = 0

    #Compute cost 
    for j in range(nu):
        w = W[j, :]
        b_j = b[0, j]
        for i in range(nm):
            if R[i, j] == 1:
                J += (np.dot(w, X[i, :]) + b_j - Y[i, j]) ** 2

    #Scale
    J = J/2
    #Regularization
    J += (lambda_/2) * (np.sum(W ** 2) + np.sum(X ** 2))

    return J

## Mean Normalization

In [2]:
def normalizeRatings(Y, R):
    """
    Mean normalize ratings per movie
    """
    nm, nu = Y.shape
    Ymean = np.zeros(nm)
    Ynorm = np.zeros_like(Y)

    for i in range(nm):
        idx = R[i, :] == 1
        if np.any(idx):
            Ymean[i] = np.mean(Y[i, idx])
            Ynorm[i, idx] = Y[i, idx] - Ymean[i]

    return Ynorm, Ymean

## Vectorized Cost Function

In [3]:
import tensorflow as tf

def cofi_cost_func_v(X, W, b, Y, R, lambda_):
    """
    Vectorized collaborative filtering cost function
    """
    pred = tf.linalg.matmul(X, tf.transpose(W)) + b
    error = (pred - Y)*R

    J = 0.5 * tf.reduce_sum(error ** 2)
    J += (lambda_ / 2) * ( tf.reduce_sum(W ** 2) + tf.reduce_sum(X ** 2))

    return J

In [8]:
# Example: 5 movies × 4 users
Y = np.array([
    [5, 4, 0, 0],
    [3, 0, 0, 5],
    [4, 0, 5, 4],
    [0, 3, 4, 0],
    [0, 0, 5, 3]
], dtype=float)

R = (Y != 0).astype(int)

num_movies, num_users = Y.shape
num_features = 3

X = tf.Variable(tf.random.normal((num_movies, num_features), dtype=tf.float64))
W = tf.Variable(tf.random.normal((num_users, num_features), dtype=tf.float64))
b = tf.Variable(tf.zeros((1, num_users), dtype=tf.float64))

Ynorm, Ymean = normalizeRatings(Y, R)

## Training Loop

In [9]:
optimizer = tf.keras.optimizers.Adam(learning_rate = 0.1)
for i in range(200):
    with tf.GradientTape() as tape:
        cost = cofi_cost_func_v(X, W, b, Ynorm, R, lambda_ = 1)
    grads = tape.gradient(cost, [X, W, b])
    optimizer.apply_gradients(zip(grads, [X, W, b]))

    if i % 20 == 0:
        print(f"Iteration {i}, cost = {cost.numpy():.2f}")

Iteration 0, cost = 49.48
Iteration 20, cost = 3.57
Iteration 40, cost = 1.82
Iteration 60, cost = 1.50
Iteration 80, cost = 1.46
Iteration 100, cost = 1.46
Iteration 120, cost = 1.45
Iteration 140, cost = 1.45
Iteration 160, cost = 1.45
Iteration 180, cost = 1.45


## Prediction

In [11]:
#Predict ratings
p = np.matmul(X.numpy(), W.numpy().T) + b.numpy()

#Restore mean
predicted_ratings = p + Ymean[:, np.newaxis]

#Recommendations for user 0
user_preds = predicted_ratings[:, 0]
top_movies = np.argsort(user_preds)[::-1]

In [12]:
print("Predicted ratings for User 0:\n")

for i in range(len(user_preds)):
    print(f"Movie {i}: Predicted Rating = {user_preds[i]:.2f}")

Predicted ratings for User 0:

Movie 0: Predicted Rating = 4.41
Movie 1: Predicted Rating = 3.45
Movie 2: Predicted Rating = 4.13
Movie 3: Predicted Rating = 3.27
Movie 4: Predicted Rating = 3.98
